# Nuclear-binding dataset anatomy audit — DevSet1010 / TestSet300 / TestSetNew46

This notebook performs **dataset/mechanism profiling only**. It does not train a predictor.

The goal is to answer, before designing a nuclear-binding model:

1. How imbalanced is `nuclear_binding` in each split?
2. How many proteins actually contain at least one nuclear-binding residue?
3. How many nuclear residues/sites occur per positive protein?
4. Which amino acids are enriched or depleted among nuclear-binding residues?
5. Are positive residues sequence-clustered or isolated?
6. How much do nuclear labels overlap with metal/small-molecule labels?
7. Is TestSet300 similar to DevSet1010, and how different is TestSetNew46?
8. Do the known baseline results indicate a distribution/hardness shift?
9. What feature mechanisms should be prioritized next?

Important: descriptive enrichment is **not causal evidence**.  
The notebook uses DevSet1010 defined by the official five bindNode24 folds.


In [ ]:
# ============================================================
# 0. DRIVE / PATH AUTO-DETECTION
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ast, json, math, os, subprocess, sys
from collections import Counter

import numpy as np
import pandas as pd

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "scipy", "scikit-learn"],
    check=True
)

from scipy.stats import fisher_exact, mannwhitneyu
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import roc_auc_score

CANDIDATE_BASES = [
    Path('/content/drive/MyDrive/Protein-binding-Phi'),
    Path('/content/drive/MyDrive/Protein-binding'),
]

BASE = None
for b in CANDIDATE_BASES:
    if (
        (b/'data/development_set/full_grouped_train_binding_sites_df.csv').exists()
        and
        (b/'data/development_set/full_grouped_test_binding_sites_df.csv').exists()
        and
        (b/'data/independent_set/grouped_test_46_new_binding_sites.csv').exists()
    ):
        BASE = b
        break

if BASE is None:
    raise FileNotFoundError(
        'Could not find the three project CSVs under Protein-binding-Phi or Protein-binding.'
    )

OUT = BASE/'results'/'NUCLEAR_DATASET_AUDIT_V1'
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

print('BASE =', BASE)
print('OUT  =', OUT)


Mounted at /content/drive
BASE = /content/drive/MyDrive/Protein-binding-Phi
OUT  = /content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_DATASET_AUDIT_V1


In [ ]:
# ============================================================
# 1. LOAD DATA + EXACT OFFICIAL DevSet1010
# ============================================================
LABELS = [
    'metal_binding',
    'nuclear_binding',
    'small_binding',
    'any_ligand_binding_sites',
]

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)

    for c in LABELS:
        if c in df.columns:
            df[c] = df[c].apply(parse_list)

    if not df.prot_id.is_unique:
        raise RuntimeError(f'{path}: duplicate prot_id')

    for r in df.itertuples():
        for c in LABELS:
            if hasattr(r, c):
                y = getattr(r, c)
                if len(y) != len(r.sequence):
                    raise RuntimeError(
                        f'{path}/{r.prot_id}/{c}: '
                        f'{len(y)} labels != {len(r.sequence)} residues'
                    )

    return df

raw_train = load_df(TRAIN_CSV)
test300 = load_df(TEST300_CSV)
test46 = load_df(TEST46_CSV)

# Locate completed bindNode24 checkout.
BN24_CANDIDATES = [
    BASE/'baseline_reproduction_master_v6'/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]

# Also check the alternate project root because older GEAR runs used Protein-binding.
other = (
    Path('/content/drive/MyDrive/Protein-binding')
    if BASE.name == 'Protein-binding-Phi'
    else Path('/content/drive/MyDrive/Protein-binding-Phi')
)
BN24_CANDIDATES += [
    other/'baseline_reproduction_master_v6'/'repos'/'bindnode24',
    other/'repos'/'bindnode24',
]

BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_dataset_audit')
    if not BN24.exists():
        subprocess.run(
            ['git', 'clone', '-q', 'https://github.com/erckert/bindNode24.git', str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
fold_ids = {}

official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]
    if seen.intersection(ids):
        raise RuntimeError(f'Fold {k}: overlap with earlier folds')
    fold_ids[k] = ids
    official_order.extend(ids)
    seen.update(ids)

official = set(official_order)
raw_ids = set(raw_train.prot_id)

missing = official - raw_ids
if missing:
    raise RuntimeError(f'Official Dev IDs missing from project CSV: {sorted(missing)[:20]}')

train_lookup = raw_train.set_index('prot_id', drop=False)
dev = train_lookup.loc[official_order].reset_index(drop=True)

assert len(dev) == 1010
assert len(test300) == 300
assert len(test46) == 46

excluded = sorted(raw_ids - official)

SETS = {
    'DevSet1010': dev,
    'TestSet300': test300,
    'TestSetNew46': test46,
}

print('Raw development lineage:', len(raw_train))
print('DevSet1010:', len(dev))
print('Excluded from DevSet1010:', len(excluded), excluded)
print('Fold sizes:', {k:len(v) for k,v in fold_ids.items()})


Raw development lineage: 1014
DevSet1010: 1010
Excluded from DevSet1010: 4 ['C8BD48', 'P84801', 'Q00277', 'Q9NZV6']
Fold sizes: {1: 202, 2: 203, 3: 203, 4: 202, 5: 200}


## 2. Core prevalence and protein-level burden

Residue prevalence alone can hide an important distinction:

- a dataset can have many positive residues concentrated in a few proteins;
- or many proteins can each contribute small interfaces.

We therefore profile both residue-level and protein-level positive burden.


In [ ]:
# ============================================================
# 2. PREVALENCE / POSITIVE-PROTEIN / BURDEN AUDIT
# ============================================================
summary_rows = []
burden_rows = []

for name, df in SETS.items():
    total_res = int(df.sequence.str.len().sum())
    all_y = np.concatenate([
        np.asarray(x, dtype=np.int8)
        for x in df.nuclear_binding
    ])

    protein_pos = np.array([
        int(np.sum(x))
        for x in df.nuclear_binding
    ])

    n_pos_proteins = int(np.sum(protein_pos > 0))

    summary_rows.append({
        'dataset': name,
        'n_proteins': len(df),
        'n_residues': total_res,
        'n_nuclear_positive_residues': int(all_y.sum()),
        'residue_positive_rate': float(all_y.mean()),
        'n_positive_proteins': n_pos_proteins,
        'positive_protein_fraction': n_pos_proteins/len(df),
        'median_positive_residues_positive_protein':
            float(np.median(protein_pos[protein_pos>0])) if n_pos_proteins else 0,
        'mean_positive_residues_positive_protein':
            float(np.mean(protein_pos[protein_pos>0])) if n_pos_proteins else 0,
        'max_positive_residues_one_protein': int(protein_pos.max()),
        'median_protein_length': float(np.median(df.sequence.str.len())),
        'mean_protein_length': float(np.mean(df.sequence.str.len())),
    })

    for pid, L, npos in zip(df.prot_id, df.sequence.str.len(), protein_pos):
        burden_rows.append({
            'dataset': name,
            'prot_id': pid,
            'length': int(L),
            'n_nuclear_positive': int(npos),
            'protein_positive': bool(npos > 0),
            'within_protein_positive_fraction': npos/max(L,1),
        })

dataset_summary = pd.DataFrame(summary_rows)
protein_burden = pd.DataFrame(burden_rows)

display(dataset_summary)

dataset_summary.to_csv(OUT/'dataset_summary.csv', index=False)
protein_burden.to_csv(OUT/'protein_burden.csv', index=False)


,dataset,n_proteins,n_residues,n_nuclear_positive_residues,residue_positive_rate,n_positive_proteins,positive_protein_fraction,median_positive_residues_positive_protein,mean_positive_residues_positive_protein,max_positive_residues_one_protein,median_protein_length,mean_protein_length
0,DevSet1010,1010,170181,2689,0.015801,108,0.106931,19.5,24.898148,112,155.0,168.496040
1,TestSet300,300,62689,1470,0.023449,66,0.220000,20.0,22.272727,64,184.5,208.963333
2,TestSetNew46,46,6227,77,0.012366,10,0.217391,6.5,7.700000,18,126.0,135.369565


## 3. Amino-acid enrichment at nuclear-binding residues

For each split, this computes:

- frequency among all residues;
- frequency among nuclear-positive residues;
- log2 enrichment;
- Fisher exact odds ratio and p-value.

The main question is whether the same residue chemistry is enriched consistently across Dev1010, Test300, and Test46.


In [ ]:
# ============================================================
# 3. POSITIVE-RESIDUE AMINO-ACID ENRICHMENT
# ============================================================
AA20 = list("ACDEFGHIKLMNPQRSTVWY")
aa_rows = []

for name, df in SETS.items():
    all_seq = ''.join(df.sequence.tolist())

    pos_aa = []
    neg_aa = []

    for r in df.itertuples():
        y = np.asarray(r.nuclear_binding, dtype=np.int8)
        seq = r.sequence

        pos_aa.extend([aa for aa,yy in zip(seq,y) if yy == 1])
        neg_aa.extend([aa for aa,yy in zip(seq,y) if yy == 0])

    all_counts = Counter(all_seq)
    pos_counts = Counter(pos_aa)
    neg_counts = Counter(neg_aa)

    n_all = sum(all_counts.values())
    n_pos = sum(pos_counts.values())
    n_neg = sum(neg_counts.values())

    for aa in AA20:
        a = pos_counts[aa]
        b = n_pos - a
        c = neg_counts[aa]
        d = n_neg - c

        odds, p = fisher_exact([[a,b],[c,d]])

        f_all = all_counts[aa]/max(n_all,1)
        f_pos = a/max(n_pos,1)

        log2_enr = np.log2((f_pos + 1e-8)/(f_all + 1e-8))

        aa_rows.append({
            'dataset': name,
            'aa': aa,
            'all_frequency': f_all,
            'positive_frequency': f_pos,
            'log2_enrichment_positive_vs_all': float(log2_enr),
            'odds_ratio_positive_vs_negative': float(odds),
            'fisher_p': float(p),
            'positive_count': int(a),
        })

aa_enrichment = pd.DataFrame(aa_rows)

for name in SETS:
    print('\n', name)
    display(
        aa_enrichment[
            aa_enrichment.dataset.eq(name)
        ].sort_values(
            'log2_enrichment_positive_vs_all',
            ascending=False
        )
    )

aa_enrichment.to_csv(OUT/'aa_enrichment.csv', index=False)



 DevSet1010


,dataset,aa,all_frequency,positive_frequency,log2_enrichment_positive_vs_all,odds_ratio_positive_vs_negative,fisher_p,positive_count
14,DevSet1010,R,0.051193,0.163258,1.673145,3.755034,5.265935e-103,439
8,DevSet1010,K,0.061840,0.143548,1.214918,2.601470,3.133966e-53,386
6,DevSet1010,H,0.024351,0.035329,0.536904,1.478333,4.003126e-04,95
11,DevSet1010,N,0.042378,0.060245,0.507523,1.458944,1.072041e-05,162
19,DevSet1010,Y,0.034322,0.043882,0.354500,1.297333,7.542117e-03,118
15,DevSet1010,S,0.062598,0.079583,0.346351,1.300846,4.059564e-04,214
13,DevSet1010,Q,0.039270,0.048345,0.299944,1.247654,1.850954e-02,130
5,DevSet1010,G,0.067428,0.081443,0.272437,1.230677,4.619288e-03,219
16,DevSet1010,T,0.053772,0.059502,0.146071,1.115309,1.816466e-01,160
18,DevSet1010,W,0.015090,0.015991,0.083692,1.061731,6.896846e-01,43



 TestSet300


,dataset,aa,all_frequency,positive_frequency,log2_enrichment_positive_vs_all,odds_ratio_positive_vs_negative,fisher_p,positive_count
34,TestSet300,R,0.056039,0.181633,1.696532,3.963902,3.034750e-66,267
28,TestSet300,K,0.059771,0.115646,0.952195,2.107297,2.574407e-16,170
26,TestSet300,H,0.027341,0.044218,0.693540,1.671260,1.813950e-04,65
35,TestSet300,S,0.060314,0.082993,0.460509,1.423732,4.602440e-04,122
36,TestSet300,T,0.051093,0.066667,0.383826,1.336876,8.223509e-03,98
39,TestSet300,Y,0.033770,0.042177,0.320715,1.267753,7.891663e-02,62
33,TestSet300,Q,0.038316,0.044898,0.228698,1.184933,1.908340e-01,66
38,TestSet300,W,0.015425,0.017687,0.197392,1.153388,4.528361e-01,26
25,TestSet300,G,0.066232,0.072109,0.122654,1.098137,3.666635e-01,106
31,TestSet300,N,0.043516,0.045578,0.066786,1.050893,6.976538e-01,67



 TestSetNew46


,dataset,aa,all_frequency,positive_frequency,log2_enrichment_positive_vs_all,odds_ratio_positive_vs_negative,fisher_p,positive_count
54,TestSetNew46,R,0.057010,0.298701,2.389422,7.463967,1.729534e-11,23
48,TestSetNew46,K,0.063112,0.181818,1.526504,3.383758,2.723235e-04,14
55,TestSetNew46,S,0.057331,0.064935,0.179683,1.143860,8.025230e-01,5
46,TestSetNew46,H,0.025052,0.025974,0.052133,1.038268,7.193896e-01,2
56,TestSetNew46,T,0.051068,0.051948,0.024652,1.018410,7.979227e-01,4
50,TestSetNew46,M,0.028906,0.025974,-0.154318,0.894682,1.000000e+00,2
53,TestSetNew46,Q,0.029549,0.025974,-0.186027,0.874432,1.000000e+00,2
45,TestSetNew46,G,0.077244,0.064935,-0.250430,0.827789,8.317332e-01,5
59,TestSetNew46,Y,0.032279,0.025974,-0.313517,0.797454,1.000000e+00,2
40,TestSetNew46,A,0.083347,0.064935,-0.360128,0.761457,6.814127e-01,5


## 4. Sequence-local context around nuclear positives

This asks whether positives sit in sequence neighborhoods enriched for:

- basic residues K/R/H;
- acidic residues D/E;
- polar residues N/Q/S/T/Y;
- aromatic residues F/W/Y/H;
- glycine/arginine-rich context.

The window analysis is descriptive and intended to motivate or reject explicit sequence-context blocks.


In [ ]:
# ============================================================
# 4. POSITIVE VS NEGATIVE SEQUENCE-NEIGHBORHOOD CONTEXT
# ============================================================
GROUPS = {
    'basic_KRH': set('KRH'),
    'strong_basic_KR': set('KR'),
    'acidic_DE': set('DE'),
    'polar_NQSTY': set('NQSTY'),
    'aromatic_FWYH': set('FWYH'),
    'gly_arg_GR': set('GR'),
}

WINDOWS = [2,4,7,10]
context_rows = []

def local_fraction(seq, i, chars, radius):
    lo = max(0, i-radius)
    hi = min(len(seq), i+radius+1)
    w = seq[lo:hi]
    return sum(aa in chars for aa in w)/max(len(w),1)

for name, df in SETS.items():
    pos_vals = {}
    neg_vals = {}

    for g in GROUPS:
        for rad in WINDOWS:
            pos_vals[(g,rad)] = []
            neg_vals[(g,rad)] = []

    for r in df.itertuples():
        seq = r.sequence
        y = np.asarray(r.nuclear_binding, dtype=np.int8)

        for i, yy in enumerate(y):
            target = pos_vals if yy else neg_vals
            for g, chars in GROUPS.items():
                for rad in WINDOWS:
                    target[(g,rad)].append(
                        local_fraction(seq, i, chars, rad)
                    )

    for g in GROUPS:
        for rad in WINDOWS:
            p = np.asarray(pos_vals[(g,rad)], dtype=float)
            n = np.asarray(neg_vals[(g,rad)], dtype=float)

            # Effect size in pooled-SD units.
            sp = np.sqrt((p.var() + n.var())/2 + 1e-12)
            smd = (p.mean()-n.mean())/sp

            context_rows.append({
                'dataset': name,
                'feature': g,
                'radius_residues': rad,
                'positive_mean': float(p.mean()),
                'negative_mean': float(n.mean()),
                'standardized_mean_difference': float(smd),
            })

context_df = pd.DataFrame(context_rows)

for name in SETS:
    print('\n', name, 'top sequence-context effects')
    display(
        context_df[
            context_df.dataset.eq(name)
        ].assign(
            abs_smd=lambda x:x.standardized_mean_difference.abs()
        ).sort_values('abs_smd', ascending=False).head(20)
    )

context_df.to_csv(OUT/'sequence_context_effects.csv', index=False)



 DevSet1010 top sequence-context effects


,dataset,feature,radius_residues,positive_mean,negative_mean,standardized_mean_difference,abs_smd
7,DevSet1010,strong_basic_KR,10,0.193859,0.111398,0.904444,0.904444
3,DevSet1010,basic_KRH,10,0.219909,0.135779,0.865650,0.865650
6,DevSet1010,strong_basic_KR,7,0.200565,0.111473,0.851918,0.851918
2,DevSet1010,basic_KRH,7,0.226693,0.135829,0.817477,0.817477
5,DevSet1010,strong_basic_KR,4,0.209679,0.111538,0.759237,0.759237
1,DevSet1010,basic_KRH,4,0.237092,0.135868,0.737445,0.737445
23,DevSet1010,gly_arg_GR,10,0.176492,0.117920,0.638514,0.638514
22,DevSet1010,gly_arg_GR,7,0.184205,0.117724,0.637143,0.637143
4,DevSet1010,strong_basic_KR,2,0.215346,0.111590,0.612885,0.612885
21,DevSet1010,gly_arg_GR,4,0.196687,0.117473,0.612472,0.612472



 TestSet300 top sequence-context effects


,dataset,feature,radius_residues,positive_mean,negative_mean,standardized_mean_difference,abs_smd
31,TestSet300,strong_basic_KR,10,0.175937,0.114201,0.745482,0.745482
27,TestSet300,basic_KRH,10,0.205377,0.141529,0.739560,0.739560
26,TestSet300,basic_KRH,7,0.213333,0.141369,0.704509,0.704509
30,TestSet300,strong_basic_KR,7,0.182442,0.114067,0.702020,0.702020
25,TestSet300,basic_KRH,4,0.224278,0.141257,0.629106,0.629106
29,TestSet300,strong_basic_KR,4,0.191811,0.113984,0.623622,0.623622
46,TestSet300,gly_arg_GR,7,0.180003,0.120907,0.612292,0.612292
47,TestSet300,gly_arg_GR,10,0.172988,0.121131,0.607153,0.607153
45,TestSet300,gly_arg_GR,4,0.188937,0.120749,0.572525,0.572525
24,TestSet300,basic_KRH,2,0.237211,0.141091,0.550365,0.550365



 TestSetNew46 top sequence-context effects


,dataset,feature,radius_residues,positive_mean,negative_mean,standardized_mean_difference,abs_smd
55,TestSetNew46,strong_basic_KR,10,0.225371,0.119163,1.284928,1.284928
54,TestSetNew46,strong_basic_KR,7,0.236305,0.118749,1.206456,1.206456
53,TestSetNew46,strong_basic_KR,4,0.264935,0.118448,1.171317,1.171317
51,TestSetNew46,basic_KRH,10,0.242476,0.144449,1.115311,1.115311
50,TestSetNew46,basic_KRH,7,0.255822,0.143999,1.042229,1.042229
49,TestSetNew46,basic_KRH,4,0.284416,0.143584,1.029340,1.029340
52,TestSetNew46,strong_basic_KR,2,0.274892,0.118347,0.937934,0.937934
48,TestSetNew46,basic_KRH,2,0.293074,0.143480,0.834840,0.834840
71,TestSetNew46,gly_arg_GR,10,0.204123,0.134028,0.641798,0.641798
70,TestSetNew46,gly_arg_GR,7,0.197206,0.133722,0.549626,0.549626


## 5. Positive clustering along the sequence

This measures whether nuclear-positive residues occur as contiguous/near-contiguous runs rather than independent points.

If positives are strongly clustered, patch-aware or sequence-neighborhood reasoning becomes more plausible.


In [ ]:
# ============================================================
# 5. POSITIVE CLUSTERING / RUN STATISTICS
# ============================================================
cluster_rows = []

def clusters_from_positions(pos, max_gap=2):
    pos = np.asarray(pos, dtype=int)
    if len(pos) == 0:
        return []

    groups = [[int(pos[0])]]
    for x in pos[1:]:
        if x - groups[-1][-1] <= max_gap:
            groups[-1].append(int(x))
        else:
            groups.append([int(x)])
    return groups

for name, df in SETS.items():
    for r in df.itertuples():
        y = np.asarray(r.nuclear_binding, dtype=np.int8)
        pos = np.flatnonzero(y)

        for gap in [1,2,3,5]:
            cs = clusters_from_positions(pos, max_gap=gap)

            cluster_rows.append({
                'dataset': name,
                'prot_id': str(r.prot_id),
                'max_gap': gap,
                'n_positive': int(len(pos)),
                'n_clusters': int(len(cs)),
                'largest_cluster_size':
                    int(max((len(c) for c in cs), default=0)),
                'mean_cluster_size':
                    float(np.mean([len(c) for c in cs])) if cs else 0.0,
                'fraction_positives_in_multiresidue_cluster':
                    float(
                        sum(len(c) for c in cs if len(c)>=2)/max(len(pos),1)
                    ),
            })

cluster_df = pd.DataFrame(cluster_rows)

cluster_summary = (
    cluster_df[
        cluster_df.n_positive.gt(0)
    ]
    .groupby(['dataset','max_gap'])
    .agg(
        positive_proteins=('prot_id','nunique'),
        mean_clusters=('n_clusters','mean'),
        mean_largest_cluster=('largest_cluster_size','mean'),
        mean_fraction_in_multiresidue_cluster=(
            'fraction_positives_in_multiresidue_cluster','mean'
        ),
    )
    .reset_index()
)

display(cluster_summary)

cluster_df.to_csv(OUT/'positive_cluster_per_protein.csv', index=False)
cluster_summary.to_csv(OUT/'positive_cluster_summary.csv', index=False)


,dataset,max_gap,positive_proteins,mean_clusters,mean_largest_cluster,mean_fraction_in_multiresidue_cluster
0,DevSet1010,1,108,11.555556,5.907407,0.715371
1,DevSet1010,2,108,7.870370,8.379630,0.838169
2,DevSet1010,3,108,5.490741,10.416667,0.914436
3,DevSet1010,5,108,4.148148,12.296296,0.958507
4,TestSet300,1,66,11.363636,5.030303,0.713888
5,TestSet300,2,66,8.121212,6.848485,0.833969
6,TestSet300,3,66,6.075758,8.151515,0.912612
7,TestSet300,5,66,5.151515,9.136364,0.947556
8,TestSetNew46,1,10,4.200000,3.000000,0.722894
9,TestSetNew46,2,10,3.900000,3.100000,0.759158


## 6. Nuclear overlap with metal and small-molecule annotations

Two overlap levels are profiled:

- **residue-level:** same residue has multiple ligand-class labels;
- **protein-level:** the same protein contains different binding-site classes.

This is important because a nuclear model should not simply learn a generic "binding-like surface" if nuclear labels have substantial cross-class overlap.


In [ ]:
# ============================================================
# 6. CROSS-LABEL OVERLAP
# ============================================================
overlap_rows = []

for name, df in SETS.items():
    nuc_all, met_all, sma_all = [], [], []
    protein_flags = []

    for r in df.itertuples():
        nuc = np.asarray(r.nuclear_binding, dtype=np.int8)
        met = np.asarray(r.metal_binding, dtype=np.int8)
        sma = np.asarray(r.small_binding, dtype=np.int8)

        nuc_all.append(nuc)
        met_all.append(met)
        sma_all.append(sma)

        protein_flags.append({
            'nuc': nuc.sum() > 0,
            'met': met.sum() > 0,
            'small': sma.sum() > 0,
        })

    nuc = np.concatenate(nuc_all).astype(bool)
    met = np.concatenate(met_all).astype(bool)
    sma = np.concatenate(sma_all).astype(bool)

    p = pd.DataFrame(protein_flags)

    overlap_rows.append({
        'dataset': name,

        'nuclear_positive_residues': int(nuc.sum()),

        'residue_nuclear_and_metal': int((nuc & met).sum()),
        'residue_nuclear_and_small': int((nuc & sma).sum()),
        'fraction_nuclear_residues_also_metal':
            float((nuc & met).sum()/max(nuc.sum(),1)),
        'fraction_nuclear_residues_also_small':
            float((nuc & sma).sum()/max(nuc.sum(),1)),

        'positive_nuclear_proteins': int(p.nuc.sum()),
        'protein_nuclear_and_metal': int((p.nuc & p.met).sum()),
        'protein_nuclear_and_small': int((p.nuc & p.small).sum()),
        'fraction_nuclear_proteins_also_metal':
            float((p.nuc & p.met).sum()/max(p.nuc.sum(),1)),
        'fraction_nuclear_proteins_also_small':
            float((p.nuc & p.small).sum()/max(p.nuc.sum(),1)),
    })

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)
overlap_df.to_csv(OUT/'cross_label_overlap.csv', index=False)


,dataset,nuclear_positive_residues,residue_nuclear_and_metal,residue_nuclear_and_small,fraction_nuclear_residues_also_metal,fraction_nuclear_residues_also_small,positive_nuclear_proteins,protein_nuclear_and_metal,protein_nuclear_and_small,fraction_nuclear_proteins_also_metal,fraction_nuclear_proteins_also_small
0,DevSet1010,2689,52,4,0.019338,0.001488,108,22,5,0.203704,0.046296
1,TestSet300,1470,65,111,0.044218,0.075510,66,27,33,0.409091,0.500000
2,TestSetNew46,77,0,1,0.000000,0.012987,10,0,1,0.000000,0.100000


## 7. Distribution shift across the three splits

We compare:

- protein length distribution;
- global amino-acid composition;
- nuclear-positive amino-acid composition;
- residue prevalence;
- positive-protein fraction.

Jensen–Shannon distance is reported for composition distributions.  
A larger distance means stronger composition shift, not necessarily greater prediction difficulty.


In [ ]:
# ============================================================
# 7. SPLIT-SHIFT ANALYSIS
# ============================================================
def aa_distribution(df, positive_only=False):
    counts = Counter()

    for r in df.itertuples():
        if not positive_only:
            counts.update(r.sequence)
        else:
            y = np.asarray(r.nuclear_binding, dtype=np.int8)
            counts.update(
                aa for aa,yy in zip(r.sequence,y)
                if yy == 1
            )

    v = np.array([counts[a] for a in AA20], dtype=float)
    return v/v.sum() if v.sum() else np.zeros_like(v)

shift_rows = []

dev_all = aa_distribution(dev, False)
dev_pos = aa_distribution(dev, True)

for name, df in SETS.items():
    all_dist = aa_distribution(df, False)
    pos_dist = aa_distribution(df, True)

    shift_rows.append({
        'dataset': name,
        'JS_global_AA_vs_Dev':
            float(jensenshannon(dev_all, all_dist, base=2)),
        'JS_positive_AA_vs_Dev':
            float(jensenshannon(dev_pos, pos_dist, base=2)),
        'median_length':
            float(np.median(df.sequence.str.len())),
        'mean_length':
            float(np.mean(df.sequence.str.len())),
    })

shift_df = pd.DataFrame(shift_rows)
display(shift_df)
shift_df.to_csv(OUT/'split_shift_summary.csv', index=False)

# Pairwise positive-AA correlation as a simple interpretability view.
pos_pivot = (
    aa_enrichment
    .pivot(index='aa', columns='dataset',
           values='log2_enrichment_positive_vs_all')
)

print('Correlation of nuclear-positive AA enrichment patterns:')
display(pos_pivot.corr())

pos_pivot.to_csv(OUT/'positive_AA_enrichment_by_split.csv')


,dataset,JS_global_AA_vs_Dev,JS_positive_AA_vs_Dev,median_length,mean_length
0,DevSet1010,0.000000,0.000000,155.0,168.496040
1,TestSet300,0.018990,0.082046,184.5,208.963333
2,TestSetNew46,0.044683,0.255069,126.0,135.369565


Correlation of nuclear-positive AA enrichment patterns:


dataset,DevSet1010,TestSet300,TestSetNew46
dataset,,,
DevSet1010,1.000000,0.936348,0.338963
TestSet300,0.936348,1.000000,0.456319
TestSetNew46,0.338963,0.456319,1.000000


## 8. Existing baseline difficulty on nuclear binding

If the completed reproduction results are present, this cell reads only the frozen result tables; it does not rerun bindEmbed21DL or bindNode24.

This helps distinguish:
- biological/data shift;
- benchmark difficulty shift.


In [ ]:
# ============================================================
# 8. READ EXISTING NUCLEAR BASELINE RESULTS
# ============================================================
result_roots = [
    BASE/'baseline_reproduction_master_v6'/'results',
    other/'baseline_reproduction_master_v6'/'results',
]

baseline_root = next(
    (p for p in result_roots
     if (p/'FULL_DIRECT_BASELINE_METRICS_300_46.csv').exists()),
    None
)

nuclear_baselines = pd.DataFrame()

if baseline_root is not None:
    z = pd.read_csv(
        baseline_root/'FULL_DIRECT_BASELINE_METRICS_300_46.csv'
    )

    if 'label' in z.columns:
        z = z[z.label.eq('nuclear_binding')].copy()

    if 'operating_point' in z.columns:
        z = z[z.operating_point.eq('fixed_0.5')].copy()

    keep = [
        c for c in [
            'method','dataset','label',
            'AP_AUPRC','AUPRC','AUROC',
            'global_positive_F1','global_MCC',
            'protein_mean_F1','protein_mean_MCC'
        ]
        if c in z.columns
    ]

    nuclear_baselines = z[keep].copy()
    display(nuclear_baselines)
    nuclear_baselines.to_csv(
        OUT/'existing_nuclear_baseline_metrics.csv',
        index=False
    )
else:
    print('Frozen baseline metric table not found; skipping this optional section.')


,method,dataset,label,AP_AUPRC,AUROC,global_positive_F1,global_MCC,protein_mean_F1,protein_mean_MCC
9,bindEmbed21DL,TestSet300,nuclear_binding,0.371532,0.921621,0.378126,0.379045,0.238822,0.224839
21,bindEmbed21DL,TestSetNew46,nuclear_binding,0.094095,0.927154,0.198582,0.239288,0.193721,0.192283
33,bindNode24,TestSet300,nuclear_binding,0.400676,0.943253,0.383162,0.386394,0.261493,0.248807
45,bindNode24,TestSetNew46,nuclear_binding,0.124657,0.948962,0.221198,0.272449,0.250441,0.254545


## 9. Automatic mechanism-oriented interpretation

The rules below are deliberately conservative.

They do not assert a biological mechanism. They identify **which hypotheses deserve explicit feature engineering** based on consistent descriptive signals across the three splits.


In [ ]:
# ============================================================
# 9. AUTOMATIC INTERPRETATION
# ============================================================
# Consistency of AA enrichment.
aa_mat = (
    aa_enrichment
    .pivot(index='aa', columns='dataset',
           values='log2_enrichment_positive_vs_all')
)

aa_mat['mean_enrichment'] = aa_mat.mean(axis=1)
aa_mat['min_enrichment'] = aa_mat[
    ['DevSet1010','TestSet300','TestSetNew46']
].min(axis=1)
aa_mat['max_enrichment'] = aa_mat[
    ['DevSet1010','TestSet300','TestSetNew46']
].max(axis=1)

consistent_enriched = (
    aa_mat[
        aa_mat.min_enrichment > 0
    ]
    .sort_values('mean_enrichment', ascending=False)
)

consistent_depleted = (
    aa_mat[
        aa_mat.max_enrichment < 0
    ]
    .sort_values('mean_enrichment')
)

print('='*88)
print('NUCLEAR DATASET MECHANISM AUDIT')
print('='*88)

print('\nConsistently enriched nuclear-positive amino acids:')
display(consistent_enriched)

print('\nConsistently depleted nuclear-positive amino acids:')
display(consistent_depleted)

# Sequence context consistency.
ctx_mean = (
    context_df
    .groupby(['feature','radius_residues'])
    .standardized_mean_difference
    .agg(['mean','min','max'])
    .reset_index()
)
ctx_mean['abs_mean'] = ctx_mean['mean'].abs()

print('\nMost consistent sequence-context effects:')
display(
    ctx_mean
    .sort_values('abs_mean', ascending=False)
    .head(20)
)

print('\nDataset shift:')
display(shift_df)

print('\nCross-label overlap:')
display(overlap_df)

# Simple feature-priority suggestions.
priority = []

def add(name, rationale, priority_level):
    priority.append({
        'feature_family': name,
        'priority': priority_level,
        'rationale_from_dataset_audit': rationale,
    })

# Basic/acidic sequence signals.
basic_rows = ctx_mean[
    ctx_mean.feature.isin(['basic_KRH','strong_basic_KR'])
]
if len(basic_rows) and basic_rows['mean'].max() > 0.10:
    add(
        'NA_ELECTRO / POSITIVE_PATCH',
        'Nuclear positives show enrichment in local basic-residue context across splits.',
        'HIGH'
    )

if (
    set(['K','R']).intersection(consistent_enriched.index)
):
    add(
        'NA_HBOND_CATIONIC',
        'K/R enrichment suggests explicit cationic/H-bond chemistry may add interpretable signal beyond ESM.',
        'HIGH'
    )

if (
    set(['F','W','Y','H']).intersection(consistent_enriched.index)
):
    add(
        'NA_AROMATIC',
        'Aromatic residue enrichment is consistent enough to justify a stacking/aromatic-patch block.',
        'MEDIUM-HIGH'
    )

# Patch/cluster rationale.
cs = cluster_summary[
    cluster_summary.max_gap.eq(2)
]
if len(cs) and cs.mean_fraction_in_multiresidue_cluster.mean() > 0.4:
    add(
        'NA_PATCH / PATCH_GRAPH',
        'A large fraction of positive residues occur in near-contiguous clusters, supporting patch-aware reasoning.',
        'HIGH'
    )
else:
    add(
        'NA_PATCH',
        'Patch features remain biologically plausible, but sequence clustering alone is not strong enough to make graph reasoning mandatory.',
        'MEDIUM'
    )

add(
    'NA_SURFACE',
    'Nucleic-acid interaction requires accessible interfaces; prior GEAR runs also showed ACCESS transfer signal. Structural confirmation should be tested directly.',
    'HIGH'
)

add(
    'EVO48',
    'Previous nuclear experiments showed complementary evolutionary signal; retain as an orthogonal biological-context branch.',
    'MEDIUM-HIGH'
)

priority_df = pd.DataFrame(priority).drop_duplicates('feature_family')
display(priority_df)

priority_df.to_csv(OUT/'recommended_feature_priorities.csv', index=False)

manifest = {
    'base': str(BASE),
    'output': str(OUT),
    'devset_definition': 'union/order of official bindNode24 five folds',
    'label': 'nuclear_binding',
    'n_sets': 3,
}

(OUT/'audit_manifest.json').write_text(json.dumps(manifest, indent=2))

print('\nSaved audit artifacts to:')
print(OUT)
print('='*88)


NUCLEAR DATASET MECHANISM AUDIT

Consistently enriched nuclear-positive amino acids:


dataset,DevSet1010,TestSet300,TestSetNew46,mean_enrichment,min_enrichment,max_enrichment
aa,,,,,,
R,1.673145,1.696532,2.389422,1.919700,1.673145,2.389422
K,1.214918,0.952195,1.526504,1.231206,0.952195,1.526504
H,0.536904,0.693540,0.052133,0.427526,0.052133,0.693540
S,0.346351,0.460509,0.179683,0.328848,0.179683,0.460509
T,0.146071,0.383826,0.024652,0.184850,0.024652,0.383826



Consistently depleted nuclear-positive amino acids:


dataset,DevSet1010,TestSet300,TestSetNew46,mean_enrichment,min_enrichment,max_enrichment
aa,,,,,,
D,-0.696758,-1.580657,-22.426431,-8.234615,-22.426431,-0.696758
C,-1.596401,-1.380090,-20.841469,-7.939320,-20.841469,-1.380090
L,-1.452903,-1.366783,-1.789169,-1.536285,-1.789169,-1.366783
I,-1.097617,-1.211878,-2.262377,-1.523957,-2.262377,-1.097617
V,-1.059046,-0.860922,-1.463365,-1.127777,-1.463365,-0.860922
E,-1.004204,-1.052449,-0.728554,-0.928402,-1.052449,-0.728554
P,-0.534901,-0.336104,-1.760496,-0.877167,-1.760496,-0.336104
A,-1.157051,-0.842474,-0.360128,-0.786551,-1.157051,-0.360128
M,-0.871251,-0.770551,-0.154318,-0.598707,-0.871251,-0.154318



Most consistent sequence-context effects:


...

,dataset,JS_global_AA_vs_Dev,JS_positive_AA_vs_Dev,median_length,mean_length
0,DevSet1010,0.000000,0.000000,155.0,168.496040
1,TestSet300,0.018990,0.082046,184.5,208.963333
2,TestSetNew46,0.044683,0.255069,126.0,135.369565



Cross-label overlap:


,dataset,nuclear_positive_residues,residue_nuclear_and_metal,residue_nuclear_and_small,fraction_nuclear_residues_also_metal,fraction_nuclear_residues_also_small,positive_nuclear_proteins,protein_nuclear_and_metal,protein_nuclear_and_small,fraction_nuclear_proteins_also_metal,fraction_nuclear_proteins_also_small
0,DevSet1010,2689,52,4,0.019338,0.001488,108,22,5,0.203704,0.046296
1,TestSet300,1470,65,111,0.044218,0.075510,66,27,33,0.409091,0.500000
2,TestSetNew46,77,0,1,0.000000,0.012987,10,0,1,0.000000,0.100000


,feature_family,priority,rationale_from_dataset_audit
0,NA_ELECTRO / POSITIVE_PATCH,HIGH,Nuclear positives show enrichment in local bas...
1,NA_HBOND_CATIONIC,HIGH,K/R enrichment suggests explicit cationic/H-bo...
2,NA_AROMATIC,MEDIUM-HIGH,Aromatic residue enrichment is consistent enou...
3,NA_PATCH / PATCH_GRAPH,HIGH,A large fraction of positive residues occur in...
4,NA_SURFACE,HIGH,Nucleic-acid interaction requires accessible i...
5,EVO48,MEDIUM-HIGH,Previous nuclear experiments showed complement...



Saved audit artifacts to:
/content/drive/MyDrive/Protein-binding-Phi/results/NUCLEAR_DATASET_AUDIT_V1
